# LC 987 — Vertical Order Traversal of Binary Tree

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Assign every node a (col, row) coordinate
during BFS, group by column, then sort each column's nodes by
(row, val) — this gives the exact vertical order including
tie-breaking by value at the same position.
</div>

## Official Problem Statement

Given the `root` of a binary tree, calculate the *vertical order
traversal* of the binary tree.

For each node at position `(row, col)`, its left and right children
will be at positions `(row+1, col-1)` and `(row+1, col+1)`
respectively.

The vertical order traversal of a binary tree is a list of
top-to-bottom orderings for each column index starting from the
leftmost column and ending at the rightmost column. There may be
multiple nodes in the same row and column. In such a case, sort
these nodes by their values.

Return the vertical order traversal of the binary tree.

**Constraints:**
- The number of nodes in the tree is in the range `[1, 1000]`
- `0 <= Node.val <= 1000`

## What This Is Actually Asking

Imagine projecting the tree onto a vertical grid where root is at
column 0. Every left child shifts one column left, every right child
shifts one right. You need to list all nodes column by column,
top to bottom within each column. When two nodes share the exact
same (row, col), sort them by value — that's the tricky tie-break
that makes this Hard.

## Walk Through an Example by Hand

```
Tree:       3
           / \
          9   20
             /  \
            15   7

BFS with (col, row, val):
  Start: (col=0, row=0, val=3)
  3's  left=9:  (col=-1, row=1, val=9)
  3's  right=20:(col=+1, row=1, val=20)
  20's left=15: (col=0,  row=2, val=15)
  20's right=7: (col=+2, row=2, val=7)

col_map:
  -1: [(1, 9)]
   0: [(0, 3), (2, 15)]
   1: [(1, 20)]
   2: [(2, 7)]

Sort each col by (row, val) (already sorted here):
  col -1: [9]
  col  0: [3, 15]
  col  1: [20]
  col  2: [7]

Answer: [[9], [3,15], [20], [7]]
```

## The Picture

```
col:  -1    0    1    2
           [3]             row 0
      [9]       [20]       row 1
           [15]      [7]   row 2
       ↑    ↑    ↑    ↑
      [9] [3,15][20] [7]

BFS queue state (col, row, node):
  Init:  [(0,0,3)]
  Pop (0,0,3)  -> push (-1,1,9), (+1,1,20)
  Pop (-1,1,9) -> no children
  Pop (+1,1,20)-> push (0,2,15), (+2,2,7)
  Pop (0,2,15) -> no children
  Pop (+2,2,7) -> no children

col_map:
  {-1:[(1,9)], 0:[(0,3),(2,15)], 1:[(1,20)], 2:[(2,7)]}
   ↑           ↑ sort by (row,val) ↑
   sorted cols: -1, 0, 1, 2
```

## When To Use This Pattern

- When the problem assigns **grid coordinates to tree nodes**,
  think **BFS with (col, row) tracking**.
- When output must be **grouped by column then sorted within
  column**, think **defaultdict(list) + sort by (row, val)**.
- When you see **left = col-1, right = col+1**, think
  **vertical traversal pattern**.
- When tie-breaking by value at same position is required,
  think **sort key = (row, val) not just row**.
- When you need to reconstruct a tree's visual column layout,
  think **this coordinate BFS approach**.

## The Approach

BFS the tree while passing (col, row) alongside each node.
Accumulate (row, val) tuples into a `defaultdict(list)` keyed
by col. After BFS, sort the column keys to iterate left-to-right,
and within each column sort the collected (row, val) pairs —
this automatically handles tie-breaking. Extract just the values
for the final output.

In [ ]:
from collections import deque, defaultdict
from typing import Optional, List


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def make_tree(vals):
    if not vals:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root

In [ ]:
def test_harness(func):
    cases = [
        # (tree_vals, expected_output)
        ([3, 9, 20, None, None, 15, 7],
         [[9], [3, 15], [20], [7]]),
        ([1, 2, 3, 4, 5, 6, 7],
         [[4], [2], [1, 5, 6], [3], [7]]),
        ([1, 2, 3, 4, 6, 5, 7],
         [[4], [2], [1, 5, 6], [3], [7]]),
        ([1],
         [[1]]),
    ]
    passed = 0
    for i, (vals, expected) in enumerate(cases):
        root = make_tree(vals)
        result = func(root)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        else:
            print(
                f"  Case {i}: got {result}, "
                f"expected {expected}"
            )
        print(f"  Case {i}: {status}")
    print(f"\nSummary: {passed}/{len(cases)} passed")

In [ ]:
def vertical_traversal(
    root: Optional[TreeNode]
) -> List[List[int]]:
    """
    Return vertical order traversal of binary tree.

    Args:
        root: root of binary tree

    Returns:
        list of columns, each column sorted by (row, val)

    Strategy:
        - BFS with (col, row) state on each node
        - Accumulate (row, val) into col_map[col]
        - Sort cols, sort each col list by (row, val)
    """
    if not root:
        pass  # return []

    col_map = defaultdict(list)
    # queue entries: (node, col, row)
    q = deque([(root, 0, 0)])

    print("[DEBUG] Starting BFS vertical traversal")

    while q:
        node, col, row = q.popleft()
        col_map[col].append((row, node.val))
        print(
            f"[DEBUG] node={node.val} col={col} row={row}"
        )
        if node.left:
            q.append((node.left, col - 1, row + 1))
        if node.right:
            q.append((node.right, col + 1, row + 1))

    print(f"[DEBUG] col_map keys: {sorted(col_map.keys())}")

    result = []
    for col in sorted(col_map.keys()):
        col_map[col].sort()  # sort by (row, val)
        result.append([val for _, val in col_map[col]])

    pass  # return result

In [ ]:
# Uncomment and run when solution is ready
# test_harness(vertical_traversal)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force (DFS all paths) | O(N^2 log N) | O(N) |
| **BFS + defaultdict + sort** | **O(N log N)** | **O(N)** |
| BFS + heap per column | O(N log N) | O(N) |

N = number of nodes. Sorting dominates at O(N log N).
col_map stores all N entries; sorted() on keys is O(W log W)
where W <= N.

## Real World Connection

Org-chart rendering at Citi maps employees to vertical swim-lanes
by department level — exactly this coordinate assignment on a tree.
AWS Cost Explorer visualizes service dependency trees by column
(service tier) to show cost attribution vertically. Data lineage
tools in DE platforms (like Apache Atlas) render DAGs in vertical
columns to show pipeline stages at the same depth. Any Gantt chart
layout or hierarchical report renderer uses the same (col, row)
coordinate BFS under the hood.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra